# 03 — Portée et closures

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- comprendre la règle **LEGB** (Local, Enclosing, Global, Built-in) ;
- lire une variable de la portée englobante dans une fonction ;
- utiliser `global` et `nonlocal` à bon escient (et savoir les éviter) ;
- écrire une **closure** : une fonction qui capture des variables d'une portée englobante.

## Prérequis

- toutes les notions précédentes ;
- **fonctions** (def) et **type hints**.

Pas encore vus :

- exceptions (prochain notebook 02_Syntaxe/06).

## Plan

1. Qu'est-ce qu'une portée ?
2. La règle LEGB
3. Lire une globale
4. Modifier une globale : `global`
5. Fonctions imbriquées et portée `enclosing`
6. Modifier une enclosing : `nonlocal`
7. Closures
8. Pièges classiques
9. Synthèse
10. Exercices

---


## 1. Qu'est-ce qu'une portée ?

Une **portée** (scope) définit où un nom est visible. Les variables déclarées dans une fonction sont **locales** : elles disparaissent à la sortie de la fonction.

In [ ]:
def f() -> None:
    x = 42
    print('dans f, x =', x)

f()

In [ ]:
# Hors de f, x n'existe pas
try:
    print(x)
except NameError as err:
    print('erreur :', err)

---


## 2. La règle LEGB

Quand Python cherche un nom, il parcourt les portées dans cet ordre :

1. **L**ocal — la fonction en cours ;
2. **E**nclosing — les fonctions englobantes, de l'intérieur vers l'extérieur ;
3. **G**lobal — le module (le fichier / notebook) ;
4. **B**uilt-in — `print`, `len`, `range`…

Le premier trouvé gagne. Si rien ne correspond, `NameError`.

---


## 3. Lire une globale

In [ ]:
TAUX: float = 0.2  # constante de module

In [ ]:
def tva(prix_ht: float) -> float:
    return prix_ht * TAUX  # lecture de TAUX depuis la portée module

tva(100.0)

Lire une globale est permis sans mot-clé particulier.

---


## 4. Modifier une globale : `global`

Sans `global`, une affectation dans une fonction crée une **nouvelle** variable locale, même si un nom identique existe au niveau module.

In [ ]:
compteur = 0

In [ ]:
def incrementer_faux() -> None:
    compteur = compteur + 1  # ⚠️ erreur : compteur est considéré local

try:
    incrementer_faux()
except UnboundLocalError as err:
    print('erreur :', err)

In [ ]:
def incrementer() -> None:
    global compteur
    compteur = compteur + 1

incrementer()
incrementer()
print(compteur)

**Usage recommandé :** éviter `global` sauf dans de petits scripts. Préférer **retourner** la nouvelle valeur.

---


## 5. Fonctions imbriquées et portée `enclosing`

On peut définir une fonction **à l'intérieur** d'une autre. La fonction interne voit les variables de la fonction externe.

In [ ]:
def externe() -> None:
    message = 'coucou'
    def interne() -> None:
        print(message)  # lu dans la portée englobante
    interne()

externe()

---


## 6. Modifier une enclosing : `nonlocal`

Pour **modifier** une variable de la fonction englobante (pas du module), on utilise `nonlocal`.

In [ ]:
def compteur_fabrique():
    n = 0
    def incrementer() -> int:
        nonlocal n
        n = n + 1
        return n
    return incrementer

c = compteur_fabrique()
print(c())
print(c())
print(c())

---


## 7. Closures

Une **closure** est une fonction qui **capture** des variables de sa portée englobante. L'exemple ci-dessus est une closure : `incrementer` capture `n`.

Autre exemple : une fabrique de fonctions de multiplication.

In [ ]:
def multiplicateur(facteur: int):
    def inner(x: int) -> int:
        return x * facteur
    return inner

double = multiplicateur(2)
triple = multiplicateur(3)
print(double(5))
print(triple(5))

Chaque closure garde **son propre** `facteur`, même après que `multiplicateur` a fini de s'exécuter.

---


## 8. Pièges classiques

### Capturer une variable de boucle

In [ ]:
fonctions = []
for i in range(3):
    def f() -> int:
        return i
    fonctions.append(f)

# Surprise : toutes renvoient 2 (la dernière valeur de i)
[fn() for fn in fonctions]

Pour capturer la valeur *au moment* de la création, on la passe comme paramètre par défaut :

In [ ]:
fonctions = []
for i in range(3):
    def f(i: int = i) -> int:
        return i
    fonctions.append(f)

[fn() for fn in fonctions]

---


## 9. Synthèse

| Situation | Outil |
|---|---|
| Lire une globale | Rien — accès direct |
| Écrire une globale | `global nom` (à éviter) |
| Lire une enclosing | Rien — accès direct |
| Écrire une enclosing | `nonlocal nom` |
| Fonction qui capture un contexte | closure |

### Règles

1. **Préférer passer / retourner** plutôt que `global`.
2. Les closures sont la manière idiomatique de « fabriquer » des fonctions configurées.
3. LEGB : Local > Enclosing > Global > Built-in.

---


## 10. Exercices

### Exercice 1 — Compteur global *(facile)*

Créer une variable `appels = 0` au niveau module. Écrire une fonction `increment() -> int` qui incrémente `appels` de 1 et renvoie sa nouvelle valeur.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Portee_et_closures", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
appels: int = 0

def increment() -> int:
    """Incrémente le compteur global d'appels."""
    global appels
    appels = appels + 1
    return appels

print(increment())
print(increment())
print(increment())
```

</details>

### Exercice 2 — Accumulateur local *(facile)*

Écrire `somme_jusqua(n: int) -> int` qui calcule `1 + 2 + ... + n`. Utiliser uniquement des variables locales.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Portee_et_closures", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
def somme_jusqua(n: int) -> int:
    """Renvoie 1+2+...+n (0 si n <= 0)."""
    total = 0
    for i in range(1, n + 1):
        total = total + i
    return total

print(somme_jusqua(10))
```

</details>

### Exercice 3 — Fabrique de puissance *(moyen)*

Écrire `puissance_n(n: int)` qui renvoie une fonction prenant `x: float` et renvoyant `x ** n`. Tester avec `carre = puissance_n(2)` puis `carre(5.0)`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Portee_et_closures", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import Callable

def puissance_n(n: int) -> Callable[[float], float]:
    """Renvoie une fonction x -> x**n."""
    def inner(x: float) -> float:
        return x ** n
    return inner

carre = puissance_n(2)
cube = puissance_n(3)
print(carre(5.0))
print(cube(5.0))
```

</details>

### Exercice 4 — Compteur encapsulé *(moyen)*

Écrire `compteur_depuis(debut: int)` qui renvoie une fonction interne `next_value() -> int`. Chaque appel de `next_value` renvoie la valeur courante puis l'incrémente. Utiliser `nonlocal`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Portee_et_closures", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import Callable

def compteur_depuis(debut: int) -> Callable[[], int]:
    """Fabrique un compteur incrémental démarrant à `debut`."""
    n = debut
    def next_value() -> int:
        nonlocal n
        valeur = n
        n = n + 1
        return valeur
    return next_value

c = compteur_depuis(10)
print(c())
print(c())
print(c())
```

</details>

### Exercice 5 — Filtre paramétré *(moyen)*

Écrire `plus_grand_que(seuil: float)` qui renvoie une fonction prenant une liste de floats et renvoyant ceux qui sont strictement supérieurs à `seuil`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Portee_et_closures", exercice=5)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import Callable

def plus_grand_que(seuil: float) -> Callable[[list[float]], list[float]]:
    """Fabrique un filtre : > seuil."""
    def inner(valeurs: list[float]) -> list[float]:
        resultat: list[float] = []
        for v in valeurs:
            if v > seuil:
                resultat.append(v)
        return resultat
    return inner

au_dessus_de_10 = plus_grand_que(10.0)
print(au_dessus_de_10([5.0, 12.0, 9.0, 18.0]))
```

</details>

### Exercice 6 — Mémoïsation simple *(difficile)*

Écrire `memoriser(fn)` qui prend une fonction `fn` de `int -> int` et renvoie une version qui met en cache les résultats dans un dictionnaire (closure). Tester avec une fonction lente (`fibonacci` récursif naïf, par exemple). Annoter avec `Callable`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Portee_et_closures", exercice=6)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import Callable

def memoriser(fn: Callable[[int], int]) -> Callable[[int], int]:
    """Renvoie une version mémoïsée de fn."""
    cache: dict[int, int] = {}
    def inner(x: int) -> int:
        if x in cache:
            return cache[x]
        resultat = fn(x)
        cache[x] = resultat
        return resultat
    return inner

def fibo(n: int) -> int:
    if n < 2:
        return n
    return fibo(n - 1) + fibo(n - 2)

fibo = memoriser(fibo)
print(fibo(30))
```

</details>

### Exercice 7 — Registre de callbacks *(difficile)*

Écrire `registre()` qui renvoie deux fonctions dans un tuple : `ajouter(cb)` et `appeler_tous()`. Les deux fonctions partagent (via closure) une liste de callbacks. `appeler_tous` les invoque dans l'ordre. Les callbacks prennent et renvoient un `int`. Afficher les retours.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Portee_et_closures", exercice=7)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import Callable

def registre() -> tuple[Callable[[Callable[[int], int]], None], Callable[[int], list[int]]]:
    """Registre de callbacks int -> int."""
    callbacks: list[Callable[[int], int]] = []

    def ajouter(cb: Callable[[int], int]) -> None:
        callbacks.append(cb)

    def appeler_tous(valeur: int) -> list[int]:
        return [cb(valeur) for cb in callbacks]

    return ajouter, appeler_tous

ajouter, appeler = registre()
ajouter(lambda x: x + 1)
ajouter(lambda x: x * 2)
print(appeler(10))
```

</details>

---


## Ressources externes

- [Portées et namespaces — tutoriel](https://docs.python.org/3/tutorial/classes.html#python-scopes-and-namespaces)
- [`nonlocal` — reference](https://docs.python.org/3/reference/simple_stmts.html#the-nonlocal-statement)
- [Closures — wiki Python](https://en.wikipedia.org/wiki/Closure_(computer_programming))

---

## Mini-exemples supplémentaires

### LEGB en action — même nom à plusieurs niveaux

In [ ]:
x = 'module'

def externe() -> None:
    x = 'enclosing'
    def interne() -> None:
        x = 'local'
        print('local   :', x)
    interne()
    print('enclosing:', x)

externe()
print('module  :', x)

### Une closure capture **la variable**, pas sa valeur

In [ ]:
def fabrique():
    n = 0
    def lire() -> int:
        return n
    def ecrire(v: int) -> None:
        nonlocal n
        n = v
    return lire, ecrire

lire, ecrire = fabrique()
print(lire())
ecrire(42)
print(lire())

### Fonction-outil pour vérifier qu'un nom est visible

In [ ]:
def existe(nom: str) -> bool:
    return nom in globals() or nom in dir(__builtins__)

print(existe('print'))
print(existe('yopyop'))

### Fonction récursive et portée

In [ ]:
def fact(n: int) -> int:
    if n <= 1:
        return 1
    return n * fact(n - 1)

fact(6)

La fonction `fact` se référence elle-même : lookup dans la portée `module`.

### Pattern : cache caché dans une closure

In [ ]:
from typing import Callable

def avec_cache(fn: Callable[[int], int]) -> Callable[[int], int]:
    cache: dict[int, int] = {}
    def wrap(x: int) -> int:
        if x not in cache:
            cache[x] = fn(x)
        return cache[x]
    return wrap

carre = avec_cache(lambda x: x * x)
print(carre(5))
print(carre(5))  # lu depuis le cache

---

## Quiz flash — vérifiez vos acquis

Ce quiz est là pour que vous vérifiiez rapidement votre compréhension avant de passer au notebook suivant. Les réponses sont dans le bloc `<details>` en dessous.


**Question 1.** Que veut dire `LEGB` ?

<details>
<summary>📖 Réponse</summary>

Local, Enclosing, Global, Built-in — l'ordre dans lequel Python résout un nom.

</details>

**Question 2.** Quelle différence entre `global` et `nonlocal` ?

<details>
<summary>📖 Réponse</summary>

`global` fait référence au niveau module ; `nonlocal` à une fonction englobante.

</details>

**Question 3.** Qu'est-ce qu'une closure ?

<details>
<summary>📖 Réponse</summary>

Une fonction qui **capture** des variables de sa portée englobante.

</details>

**Question 4.** Pourquoi `global` est-il déconseillé ?

<details>
<summary>📖 Réponse</summary>

Parce qu'il crée un couplage invisible : lire un code qui modifie des globales est difficile. Préférer passer et retourner.

</details>

**Question 5.** Que fait `nonlocal` si la variable n'existe pas dans la portée englobante ?

<details>
<summary>📖 Réponse</summary>

Python lève `SyntaxError` à l'analyse — c'est une erreur de compilation, pas d'exécution.

</details>